In [ ]:
import numpy as np
import scipy.linalg as nla
import pandas as pd

In [2]:
matrice_ad = pd.read_csv("adjacency_matrix.csv")
id_site = matrice_ad["Unnamed: 0"].to_list() # la liste des noms de page de la matrice
id_site.append("Virtual node NULL")
matrice_ad = matrice_ad.drop(["Unnamed: 0"], axis=1)
matrice_ad = matrice_ad.to_numpy()

In [3]:
# Crée un noeud viruel NULL  relier aux noeuds terminaux du graphe
def handle_row_zeros(mat):
    sum_row = np.sum(mat, axis=1)
    n = mat.shape[0]
    res = np.zeros((n+1,n+1))
    res[:n,:n ]= mat.copy()
    for i in range(n):
        if  sum_row[i] == 0:
            res[i,n] = 1
            res[n,i] = 1
    return res 

In [4]:
# Retourne le matrice de transition P associée à la matrice d'adjacence
def matrice_transition(mat):
    res = handle_row_zeros(mat)
    return (1 / np.sum(res, axis=1)) * np.transpose(res)

In [5]:
# Applique l'algorithme du PageRank à une matrice de transition P donnée en appliquant la méthode de la puissance
# personalized_nodes contient la liste des noeuds cibles pour exécuter le personalised PageRank
# Si personalized_nodes est vide c'est l'algorithme du PageRank basique qui est utilisé
def PageRank(P, personalized_nodes = [], beta = 0.85, epsilon = 1e-10):
    N = P.shape[0]
    L = len(personalized_nodes)
    if L == 0:
        q0 = (1 / np.sqrt(N)) * np.ones(N)
        v = np.ones(N)
        L = N
    else: 
        v = np.zeros(N)
        for node in personalized_nodes:
            v[node] = 1 
        q0 = v
    error = 1
    while error > epsilon:
        q1 = beta * np.dot(P, q0) 
        q1 = q1 + ((1 - beta) / L) * np.sum(q0) * v
        q1 = (1 / nla.norm(q1, 1)) * q1
        error = nla.norm(q1 - q0, 2)
        q0 = q1
    return q0   

In [6]:
# Prend en paramètre la liste des noms de site web id_site et le vecteur retourner par la fonction PageRank ranking

# Associe chaque site a son score PageRank et les tris par ordre décroisant
def liste_ranking(id_site,ranking):
    n = len(id_site) + 1
    resultat = {"id_site" : id_site,
                "ranking" : ranking}
    resultat = pd.DataFrame(resultat)
    resultat = resultat.sort_values(by="ranking", ascending=False)
    resultat.insert(0, "classement", [i for i in range(1,n)])
    return resultat

# Résultats du PageRank et du personalised Pagerank

In [7]:
P = matrice_transition(matrice_ad)
ranking = PageRank(P,beta=0.85)
ranking = liste_ranking(id_site, ranking)
ranking.head(20)

,classement,id_site,ranking
24,1,United_States,0.016754
9,2,Europe,0.009717
302,3,United_Kingdom,0.009124
84,4,England,0.007252
64,5,World_War_II,0.006912
63,6,France,0.006869
231,7,Germany,0.005427
160,8,English_language,0.005205
6,9,Africa,0.004883
59,10,India,0.004364


In [15]:
ranking = PageRank(P, personalized_nodes=[id_site.index("Periodic_table")],beta=0.85)
ranking = liste_ranking(id_site, ranking)
ranking.head(20)

,classement,id_site,ranking
122,1,Periodic_table,0.162613
125,2,Chemical_element,0.012597
24,3,United_States,0.010609
322,4,Russia,0.006809
513,5,Metal,0.005955
701,6,List_of_elements_by_name,0.005648
9,7,Europe,0.005374
231,8,Germany,0.005093
302,9,United_Kingdom,0.004728
233,10,Earth,0.004721


In [16]:
ranking = PageRank(P, personalized_nodes=[id_site.index("Vegetable")],beta=0.85)
ranking = liste_ranking(id_site, ranking)
ranking.head(20)

,classement,id_site,ranking
405,1,Vegetable,0.151680
24,2,United_States,0.013719
406,3,Plant,0.009220
160,4,English_language,0.009203
9,5,Europe,0.008367
439,6,Human,0.008174
529,7,Agriculture,0.008009
404,8,Fruit,0.007860
1045,9,Vitamin,0.007123
600,10,Vitamin_C,0.006909


# Résultats du  reverse PageRank et du  reverse personalised Pagerank

In [9]:
matrice_ad_T = np.transpose(matrice_ad)
P_reverse = matrice_transition(matrice_ad_T)
ranking = PageRank(P_reverse ,beta=0.85)
ranking = liste_ranking(id_site, ranking)
ranking.head(20)

,classement,id_site,ranking
4169,1,Virtual node NULL,0.095656
24,2,United_States,0.004968
84,3,England,0.003367
302,4,United_Kingdom,0.003110
6,5,Africa,0.002871
122,6,Periodic_table,0.002808
421,7,Dinosaur,0.002266
701,8,List_of_elements_by_name,0.002126
85,9,London,0.001929
56,10,19th_century,0.001887


In [10]:
ranking = PageRank(P_reverse , personalized_nodes=[190,22],beta=0.85)
ranking = liste_ranking(id_site, ranking)
ranking.head(20)

,classement,id_site,ranking
22,1,John_F._Kennedy,0.078179
190,2,Lisbon,0.075230
4169,3,Virtual node NULL,0.053771
4,4,Atlantic_Ocean,0.010660
9,5,Europe,0.010422
298,6,Trojan_War,0.010010
627,7,Global_city,0.009754
332,8,Christopher_Columbus,0.009714
189,9,Portugal,0.009709
2022,10,Goa,0.009425
